# Laboratorio 1: Introducción a Redes Neuronales Convolucionales (CNN)
## Clasificación de Botellas: Glass vs Plastic

### Objetivos
- Entender qué es una CNN y cómo funciona cada capa
- Construir y entrenar una CNN básica para clasificación de imágenes
- Interpretar las métricas de entrenamiento (accuracy, loss)
- Visualizar la matriz de confusión

### Estructura del laboratorio
1. Preparación de datos (división train/validation)
2. Visualización interactiva del dataset
3. Construcción de la CNN (explicada capa por capa)
4. Entrenamiento y análisis de resultados
5. Predicciones y exportación del modelo


In [ ]:
# Verificar que las dependencias están instaladas
# (Si falla, ejecute 'uv sync' en la terminal antes de abrir el notebook)
import tensorflow as tf
print(f"TensorFlow {tf.__version__} — OK")

In [ ]:
import os
import shutil
import random
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

import ipywidgets as widgets
from IPython.display import display, clear_output

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print("Librerías cargadas correctamente")

---
## 1. Preparación de Datos

Dividimos el dataset de botellas en **train (80%)** y **validation (20%)**.
- **Train**: el modelo aprende de estas imágenes
- **Validation**: se usa para evaluar el modelo (el modelo **nunca** ve estas imágenes durante el entrenamiento)


In [ ]:
def split_dataset(dataset_dir, output_dir, split_ratio=0.8):
    """Divide el dataset en train y validation."""
    train_dir = os.path.join(output_dir, 'train')
    validation_dir = os.path.join(output_dir, 'validation')

    if os.path.exists(train_dir) and os.path.exists(validation_dir):
        print(f"La división ya existe en '{output_dir}'. Usando datos existentes.")
        return

    print(f"Creando división en '{output_dir}'...")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(validation_dir, exist_ok=True)

    categories = ['glass', 'plastic']
    random.seed(42)

    for category in categories:
        category_dir = os.path.join(dataset_dir, category)
        images = os.listdir(category_dir)
        random.shuffle(images)

        train_size = int(len(images) * split_ratio)

        for subset, img_list in [('train', images[:train_size]),
                                  ('validation', images[train_size:])]:
            dest = os.path.join(output_dir, subset, category)
            os.makedirs(dest, exist_ok=True)
            for img in img_list:
                shutil.copy(os.path.join(category_dir, img), os.path.join(dest, img))

        print(f"  {category}: {train_size} train, {len(images) - train_size} validation")

    print("División completada.")

dataset_dir = '../data/botellas'
output_dir = '../data/botellas_split'
split_dataset(dataset_dir, output_dir, split_ratio=0.8)

## 2. Exploración Interactiva del Dataset

Use este widget para navegar las imágenes. Observe las diferencias entre botellas de vidrio y plástico.


In [ ]:
def explorar_botellas():
    """Widget para explorar el dataset de botellas."""
    clase_dd = widgets.Dropdown(options=['glass', 'plastic'], description='Tipo:')
    subset_dd = widgets.Dropdown(options=['train', 'validation'], description='Set:')
    btn = widgets.Button(description='Mostrar otras', icon='refresh', button_style='info')
    output = widgets.Output()

    def mostrar(_=None):
        with output:
            clear_output(wait=True)
            img_dir = os.path.join(output_dir, subset_dd.value, clase_dd.value)
            imgs = [os.path.join(img_dir, f) for f in os.listdir(img_dir)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if not imgs:
                print("No hay imágenes")
                return

            sample = random.sample(imgs, min(8, len(imgs)))
            fig, axes = plt.subplots(2, 4, figsize=(16, 8))
            for ax in axes.ravel():
                ax.axis('off')
            for ax, path in zip(axes.ravel(), sample):
                img = load_img(path, target_size=(64, 64))
                ax.imshow(img)

            plt.suptitle(f'{clase_dd.value} — {subset_dd.value} ({len(imgs)} imágenes)',
                         fontsize=14)
            plt.tight_layout()
            plt.show()

    for w in [clase_dd, subset_dd]:
        w.observe(lambda c: mostrar(), names='value')
    btn.on_click(mostrar)
    display(widgets.HBox([clase_dd, subset_dd, btn]))
    display(output)
    mostrar()

explorar_botellas()

---
## 3. Construcción de la CNN

Una CNN procesa imágenes en etapas. Cada capa tiene una función específica:

| Capa | ¿Qué hace? | Analogía |
|---|---|---|
| `Conv2D(32, 3×3)` | Detecta **patrones** (bordes, texturas) usando 32 filtros | Un lente que resalta detalles |
| `MaxPooling2D(2×2)` | Reduce la imagen a la mitad, conservando lo importante | Hacer zoom out |
| `Flatten` | Convierte la imagen 2D en un vector 1D | Aplanar una hoja |
| `Dense(64)` | Combina toda la información para tomar decisiones | El "cerebro" |
| `Dense(2, softmax)` | Salida: probabilidad de cada clase | La respuesta final |


In [ ]:
# Crear el modelo CNN
model = Sequential([
    # Capa 1: Detecta 32 patrones básicos (bordes, líneas)
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(64, 64, 3)),
    MaxPooling2D(pool_size=(2, 2)),   # 64×64 → 32×32

    # Capa 2: Detecta 32 patrones más complejos (formas, texturas)
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2, 2)),   # 32×32 → 16×16

    # Clasificador
    Flatten(),                         # 16×16×32 = 8192 valores
    Dense(64, activation='relu'),      # Combina la información
    Dense(2, activation='softmax')     # 2 clases: glass, plastic
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

### ¿Qué "ve" cada capa?

Ejecute esta celda para visualizar cómo la CNN transforma la imagen en cada etapa.


In [ ]:
def visualizar_capas():
    """Muestra los mapas de características de cada capa convolucional."""
    img_dir = os.path.join(output_dir, 'train', 'glass')
    imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not imgs:
        print("No hay imágenes disponibles")
        return

    img_path = os.path.join(img_dir, imgs[0])
    img = load_img(img_path, target_size=(64, 64))
    img_array = img_to_array(img) / 255.0
    img_batch = np.expand_dims(img_array, 0)

    # Obtener capas convolucionales
    conv_names = [layer.name for layer in model.layers if 'conv' in layer.name]

    fig, axes = plt.subplots(1 + len(conv_names), 8, figsize=(20, 4 * (1 + len(conv_names))))

    # Imagen original
    axes[0][0].imshow(img)
    axes[0][0].set_title("Imagen original", fontsize=12, fontweight='bold')
    for j in range(1, 8):
        axes[0][j].axis('off')
    axes[0][0].axis('off')

    # Pasar datos capa por capa para obtener salidas intermedias
    x = img_batch
    conv_idx = 0
    for layer in model.layers:
        x = layer(x, training=False)
        if 'conv' in layer.name:
            feature_maps = x.numpy()
            for j in range(8):
                if j < feature_maps.shape[-1]:
                    axes[conv_idx+1][j].imshow(feature_maps[0, :, :, j], cmap='viridis')
                    axes[conv_idx+1][j].set_title(f'Filtro {j+1}', fontsize=9)
                axes[conv_idx+1][j].axis('off')

            axes[conv_idx+1][0].set_ylabel(
                f'{layer.name}\n{feature_maps.shape[1]}x{feature_maps.shape[2]}',
                fontsize=10, rotation=0, labelpad=80)
            conv_idx += 1

    plt.suptitle('¿Qué detecta cada capa de la CNN?', fontsize=16)
    plt.tight_layout()
    plt.show()

visualizar_capas()

**Observe:** Las primeras capas detectan bordes y texturas simples.
Las capas más profundas combinan esos patrones en formas más complejas.


---
## 4. Acondicionamiento de Datos

Redimensionamos las imágenes a **64×64 píxeles** y las **normalizamos** (valores entre 0 y 1).


In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)

train_set = train_datagen.flow_from_directory(
    os.path.join(output_dir, 'train'),
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical'
)

validation_set = validation_datagen.flow_from_directory(
    os.path.join(output_dir, 'validation'),
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical'
)

print(f"\nClases: {train_set.class_indices}")

---
## 5. Entrenamiento

Entrenamos el modelo por **100 épocas**. Una época = una pasada completa por todo el dataset.


In [ ]:
history = model.fit(
    train_set,
    steps_per_epoch=train_set.samples // train_set.batch_size,
    epochs=100,
    validation_data=validation_set,
    validation_steps=validation_set.samples // validation_set.batch_size
)

loss, accuracy = model.evaluate(validation_set)
print(f'\nResultado final → Loss: {loss:.4f}, Accuracy: {accuracy:.4f}')

In [ ]:
# Curvas de entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history.history['accuracy']) + 1)

ax1.plot(epochs_range, history.history['accuracy'], 'b-', label='Entrenamiento', linewidth=2)
ax1.plot(epochs_range, history.history['val_accuracy'], 'r-', label='Validación', linewidth=2)
ax1.set_title('Precisión (Accuracy)')
ax1.set_xlabel('Época')
ax1.set_ylabel('Precisión')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history.history['loss'], 'b-', label='Entrenamiento', linewidth=2)
ax2.plot(epochs_range, history.history['val_loss'], 'r-', label='Validación', linewidth=2)
ax2.set_title('Pérdida (Loss)')
ax2.set_xlabel('Época')
ax2.set_ylabel('Pérdida')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Analice el resultado y comente:**
- ¿El modelo converge? ¿La accuracy sube y el loss baja?
- ¿Hay diferencia entre las curvas de entrenamiento y validación?
- Si la curva de validación se estanca o empeora mientras la de entrenamiento sigue mejorando,
  eso se llama **overfitting** (el modelo memoriza en vez de aprender).

**Respuesta:** *(Escriba aquí)*


---
## 6. Matriz de Confusión

La matriz de confusión muestra **dónde se equivoca** el modelo:
- Diagonal: predicciones correctas
- Fuera de la diagonal: errores


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

validation_set.reset()
predictions = model.predict(validation_set, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = validation_set.classes
class_names = list(validation_set.class_indices.keys())

# Reporte
print("Reporte de Clasificación:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

# Matriz visual
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión")
plt.tight_layout()
plt.show()

---
## 7. Predicciones Interactivas

Use este widget para probar el modelo con imágenes del set de validación.


In [ ]:
class_indices = train_set.class_indices
class_labels = {v: k for k, v in class_indices.items()}

def predecir_imagen(img_path, model, class_labels):
    """Predice la clase de una imagen y muestra la confianza."""
    img = load_img(img_path, target_size=(64, 64))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)
    pred_idx = np.argmax(prediction)
    label = class_labels[pred_idx]
    confidence = float(prediction[0][pred_idx])
    return label, confidence, prediction[0]

def predictor_interactivo():
    """Widget para probar predicciones una por una."""
    btn = widgets.Button(description='Predecir imagen aleatoria',
                         icon='search', button_style='success')
    output = widgets.Output()

    val_dir = os.path.join(output_dir, 'validation')
    all_imgs = []
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        all_imgs += glob.glob(os.path.join(val_dir, '*', ext))

    def predecir(_=None):
        with output:
            clear_output(wait=True)
            if not all_imgs:
                print("No hay imágenes de validación")
                return

            img_path = random.choice(all_imgs)
            clase_real = os.path.basename(os.path.dirname(img_path))
            label, conf, probs = predecir_imagen(img_path, model, class_labels)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4),
                                            gridspec_kw={'width_ratios': [1, 1.2]})

            ax1.imshow(load_img(img_path, target_size=(64, 64)))
            ok = label == clase_real
            color = 'green' if ok else 'red'
            simbolo = '✓' if ok else '✗'
            ax1.set_title(f'{simbolo} Real: {clase_real} | Pred: {label}',
                          fontsize=12, color=color, fontweight='bold')
            ax1.axis('off')

            names = list(class_labels.values())
            colors = ['#2ecc71' if i == pred_idx else '#bdc3c7'
                      for i, pred_idx in enumerate(range(len(names)))]
            colors = ['#2ecc71' if n == label else '#bdc3c7' for n in names]
            bars = ax2.barh(names, probs, color=colors)
            ax2.set_xlim(0, 1)
            ax2.set_xlabel('Confianza')
            for bar, val in zip(bars, probs):
                ax2.text(bar.get_width() + 0.02,
                         bar.get_y() + bar.get_height() / 2,
                         f'{val:.1%}', va='center')
            plt.tight_layout()
            plt.show()

    btn.on_click(predecir)
    display(btn)
    display(output)
    predecir()

predictor_interactivo()

In [ ]:
# Muestra 20 predicciones en grilla
val_dir_path = os.path.join(output_dir, 'validation')
image_paths = []
for ext in ('*.jpg', '*.jpeg', '*.png'):
    image_paths += glob.glob(os.path.join(val_dir_path, '*', ext))

sample_images = random.sample(image_paths, min(20, len(image_paths)))

plt.figure(figsize=(15, 12))
for i, img_path in enumerate(sample_images):
    label, conf, _ = predecir_imagen(img_path, model, class_labels)
    real = os.path.basename(os.path.dirname(img_path))
    img = load_img(img_path, target_size=(64, 64))

    plt.subplot(4, 5, i + 1)
    plt.imshow(img)
    plt.axis("off")
    color = 'green' if label == real else 'red'
    plt.title(f"{label} ({conf:.0%})", fontsize=9, color=color)

plt.suptitle('Predicciones del modelo (verde=correcto, rojo=error)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Exportar Modelo

Guardamos el modelo para usarlo en el siguiente laboratorio.


In [ ]:
model.save("./modelo_lab1.h5")
print("Modelo exportado como: modelo_lab1.h5")
print("Este modelo se usará en el Lab 2 como punto de comparación.")